In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS earthquake_data_conn
  TYPE HTTP
  OPTIONS (
    host = 'https://earthquake.usgs.gov',
    port = 443,
    base_path = '/earthquakes/feed/v1.0/',
    bearer_token = 'na'
  )

In [0]:

%python
#Hardcode Base URL
from databricks.sdk import WorkspaceClient
ws = WorkspaceClient()

conn = ws.connections.get("earthquake_data_conn")
#print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
%python
dbutils.widgets.text('catalog_name','geo_proj','geo_proj')
catalog_name=dbutils.widgets.get('catalog_name')
print(catalog_name)

In [0]:
%python

spark.sql(f"use catalog {catalog_name}") 
spark.sql(f"use schema bronze") 
       
#Create Volume
#use catalog geo_proj;
# use schema bronze;
spark.sql("create volume if not exists earthquake_data")


In [0]:
%python
import requests
import json
import datetime


url = f"{base_url}summary/all_month.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Request failed with status code {response.status_code}")
data=response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)